In [2]:
!pip install -U -q "mineru[core]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.6/786.6 kB 16.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.2/16.2 MB 90.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.3/98.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 112.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import os
from pathlib import Path

INPUT_DIR = Path("/kaggle/input/datasets/octave2402/filepdf")

OUTPUT_DIR = Path("/kaggle/working/md_files")
TMP_DIR = Path("/kaggle/temp/pdf_tmp")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(INPUT_DIR.rglob("*.pdf")) if INPUT_DIR.exists() else []
print(len(pdfs))


28


In [ ]:
import subprocess
import shutil
import time
from pathlib import Path

log_file = OUTPUT_DIR / "run.log"

print("⏳ Inizio la conversione con MinerU (Testo + Immagini + Formule)...\n")
print("-" * 50)

for pdf in pdfs:
    # 1. Salta i file di servizio
    if pdf.name.startswith("0_"):
        print(f"SKIP {pdf.name} (file 0_ escluso)")
        continue

    # 2. Ricrea la struttura logica delle cartelle originali
    relative_pdf = pdf.relative_to(INPUT_DIR)
    target_dir = OUTPUT_DIR / relative_pdf.parent
    target_dir.mkdir(parents=True, exist_ok=True)

    name = pdf.stem
    
    # 3. Percorso di output di MinerU AGGIORNATO
    # Abbiamo aggiunto "hybrid_auto" nel percorso per far combaciare la ricerca
    out_md = target_dir / name / "hybrid_auto" / f"{name}.md"

    # Ora lo SKIP funzionerà alla perfezione!
    if out_md.exists():
        print(f"SKIP {relative_pdf} (già convertito)")
        continue

    # 4. Copia temporanea su disco veloce/scrivibile
    src = TMP_DIR / relative_pdf
    src.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pdf, src)

    t0 = time.time()

    # 5. Esecuzione MinerU
    r = subprocess.run(
        [
            "mineru", 
            "-p", str(src),
            "-o", str(target_dir),
            "-m", "auto"
        ],
        capture_output=True, text=True
    )

    elapsed = time.time() - t0
    
    # Adesso anche il check finale di "OK" troverà il file giusto
    ok = (r.returncode == 0 and out_md.exists())

    msg = f"{relative_pdf} OK {elapsed:.0f}s" if ok else f"{relative_pdf} FAIL {r.returncode}"
    print(msg)

    with open(log_file, "a") as f:
        f.write(msg + "\n")

    if not ok:
        print("--- ERRORE STDERR ---", r.stderr[-1000:])

    # 6. Pulizia del file temporaneo
    src.unlink(missing_ok=True)
    
print("\n✅ ELABORAZIONE COMPLETATA! Markdown e Immagini sono in:", OUTPUT_DIR)

⏳ Inizio la conversione con MinerU (Testo + Immagini + Formule)...

--------------------------------------------------
SKIP 0_indice_analitico.pdf (file 0_ escluso)
SKIP 0_indice_generale.pdf (file 0_ escluso)
SKIP Probability,_Random_Variables_and_Stochastic_Processes/10_Random_Walks_and_Other_Applications.pdf (già convertito)
SKIP Probability,_Random_Variables_and_Stochastic_Processes/11_Spectral_Representation.pdf (già convertito)
SKIP Probability,_Random_Variables_and_Stochastic_Processes/12_Spectrum_Estimation.pdf (già convertito)
SKIP Probability,_Random_Variables_and_Stochastic_Processes/13_Mean_Square_Estimation.pdf (già convertito)
SKIP Probability,_Random_Variables_and_Stochastic_Processes/14_Entropy.pdf (già convertito)
SKIP Probability,_Random_Variables_and_Stochastic_Processes/15_Markov_Chains.pdf (già convertito)


In [ ]:
import shutil

print("Sto comprimendo l'intero pacchetto (Markdown + Immagini ritagliate)...")
shutil.make_archive("/kaggle/working/risultati_mineru", 'zip', "/kaggle/working/md_files")
print("✓ Fatto! Vai nel pannello di destra 'Output' per scaricare: risultati_mineru.zip")

In [ ]:
import os
from IPython.display import FileLink

# Assicuriamoci di essere nella cartella corretta
os.chdir('/kaggle/working')

# Genera un link cliccabile per il download
display(FileLink('risultati_mineru.zip'))